# 02 — Limpeza dos dados do ClinicalTrials.gov
Este notebook transforma os dados brutos (extração de 12/10/2026) em tabelas limpas para análise.


## 1. Ler os dados brutos

In [1]:
import json
import pandas as pd

In [2]:
with open("../data/raw/estudos_obesidade_2026-10-12.json", encoding="utf-8") as ficheiro:
    estudos_brutos = json.load(ficheiro)

In [3]:
len(estudos_brutos)

8431

In [4]:
print(json.dumps(estudos_brutos[0], indent=2, ensure_ascii=False))

{
  "protocolSection": {
    "identificationModule": {
      "nctId": "NCT06151145",
      "briefTitle": "Stepped Care for Weight Loss Maintenance"
    },
    "statusModule": {
      "overallStatus": "ACTIVE_NOT_RECRUITING",
      "startDateStruct": {
        "date": "2024-02-01",
        "type": "ACTUAL"
      },
      "primaryCompletionDateStruct": {
        "date": "2028-06",
        "type": "ESTIMATED"
      },
      "completionDateStruct": {
        "date": "2028-06",
        "type": "ESTIMATED"
      }
    },
    "sponsorCollaboratorsModule": {
      "leadSponsor": {
        "class": "OTHER"
      }
    },
    "designModule": {
      "studyType": "INTERVENTIONAL",
      "phases": [
        "NA"
      ],
      "enrollmentInfo": {
        "count": 192
      }
    },
    "armsInterventionsModule": {
      "interventions": [
        {
          "type": "BEHAVIORAL"
        },
        {
          "type": "BEHAVIORAL"
        },
        {
          "type": "BEHAVIORAL"
        }
      

In [5]:
print(json.dumps(estudos_brutos[0]["protocolSection"]["designModule"], indent=2))

{
  "studyType": "INTERVENTIONAL",
  "phases": [
    "NA"
  ],
  "enrollmentInfo": {
    "count": 192
  }
}


In [6]:
for estudo in estudos_brutos:
    status = estudo["protocolSection"]["statusModule"]
    if status["overallStatus"] == "TERMINATED":
        print(json.dumps(status, indent=2))
        break

{
  "overallStatus": "TERMINATED",
  "whyStopped": "difficulty in recruiting",
  "startDateStruct": {
    "date": "2013-12",
    "type": "ACTUAL"
  },
  "primaryCompletionDateStruct": {
    "date": "2022-01",
    "type": "ACTUAL"
  },
  "completionDateStruct": {
    "date": "2022-06",
    "type": "ACTUAL"
  }
}


## 2. Construir a tabela estudos

In [7]:
linhas = []
for estudo in estudos_brutos:
   
    protocolo = estudo.get("protocolSection", {})

    
    ident = protocolo.get("identificationModule", {})
    status =protocolo.get("statusModule",{})
    design = protocolo.get("designModule",{})
    patrocinio = protocolo.get("sponsorCollaboratorsModule",{})

    
    inicio = status.get("startDateStruct", {})
    fim_primaria = status.get("primaryCompletionDateStruct",{})
    fim = status.get("completionDateStruct",{})
    inscricao = design.get("enrollmentInfo",{})
    patrocinador = patrocinio.get("leadSponsor",{})

   
    fases = design.get("phases",[])

    linhas.append({
        "nct_id": ident.get("nctId"),
        "titulo": ident.get("briefTitle"),
        "tipo_estudo": design.get("studyType"),
        "estado": status.get("overallStatus"),
        "motivo_paragem": status.get("whyStopped"),
        "data_inicio": inicio.get("date"),
        "tipo_data_inicio": inicio.get("type"),
        "data_fim_primaria": fim_primaria.get("date"),
        "tipo_data_fim_primaria": fim_primaria.get("type"),
        "data_fim": fim.get("date"),
        "tipo_data_fim": fim.get("type"),
        "participantes": inscricao.get("count"),
        "fase": "/".join(fases) if fases else None,
        "classe_patrocinador": patrocinador.get("class"),
    })

df_estudos = pd.DataFrame(linhas)

In [8]:
df_estudos.shape

(8431, 14)

In [9]:
df_estudos.head()

,nct_id,titulo,tipo_estudo,estado,motivo_paragem,data_inicio,tipo_data_inicio,data_fim_primaria,tipo_data_fim_primaria,data_fim,tipo_data_fim,participantes,fase,classe_patrocinador
0,NCT06151145,Stepped Care for Weight Loss Maintenance,INTERVENTIONAL,ACTIVE_NOT_RECRUITING,NaN,2024-02-01,ACTUAL,2028-06,ESTIMATED,2028-06,ESTIMATED,192.0,NA,OTHER
1,NCT02185872,CrossFit Exercise to Improve Glucose Control f...,INTERVENTIONAL,COMPLETED,NaN,2012-02,NaN,2012-05,ACTUAL,2012-05,ACTUAL,23.0,NA,OTHER
2,NCT03917901,Computer-Delivered Intervention for Individual...,INTERVENTIONAL,COMPLETED,NaN,2019-09-16,ACTUAL,2021-03-06,ACTUAL,2021-03-06,ACTUAL,131.0,NA,OTHER
3,NCT01049698,Improving Muscle for Functional Independence T...,INTERVENTIONAL,COMPLETED,NaN,2010-02,NaN,2014-05,ACTUAL,2014-05,ACTUAL,126.0,NA,OTHER
4,NCT06989177,Probiotic Intervention on Body Weight,INTERVENTIONAL,RECRUITING,NaN,2025-06-10,ACTUAL,2026-12-30,ESTIMATED,2026-12-31,ESTIMATED,140.0,NA,OTHER_GOV


In [10]:
df_estudos.isna().sum()

nct_id                       0
titulo                       0
tipo_estudo                  0
estado                       0
motivo_paragem            7935
data_inicio                  0
tipo_data_inicio          2062
data_fim_primaria            3
tipo_data_fim_primaria       4
data_fim                    54
tipo_data_fim               57
participantes                2
fase                         3
classe_patrocinador          0
dtype: int64

In [11]:
parados = df_estudos["estado"].isin(["TERMINATED", "SUSPENDED", "WITHDRAWN"])
df_estudos[parados]["motivo_paragem"].isna().sum()

np.int64(40)

In [12]:
df_estudos["nct_id"].is_unique


True

In [13]:
df_estudos["tipo_estudo"].value_counts(dropna=False)

tipo_estudo
INTERVENTIONAL    8431
Name: count, dtype: int64

In [14]:
sem_tipo = df_estudos["tipo_data_inicio"].isna()
df_estudos[sem_tipo]["data_inicio"].str[:4].value_counts().sort_index()

data_inicio
2010    325
2011    317
2012    353
2013    323
2014    280
2015    263
2016    193
2017      8
Name: count, dtype: int64

In [15]:
df_estudos["tipo_data_inicio"].isna().mean()

np.float64(0.24457359743802634)

In [16]:
parados.sum()

np.int64(536)

In [17]:
df_estudos[~parados]["motivo_paragem"].notna().sum()

np.int64(0)

In [18]:
(df_estudos["data_fim"].notna() & df_estudos["tipo_data_fim"].isna()).sum()

np.int64(3)

In [19]:
df_estudos["participantes"]= df_estudos["participantes"].astype("Int64")

In [20]:
df_estudos.info()

<class 'pandas.DataFrame'>
RangeIndex: 8431 entries, 0 to 8430
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   nct_id                  8431 non-null   str  
 1   titulo                  8431 non-null   str  
 2   tipo_estudo             8431 non-null   str  
 3   estado                  8431 non-null   str  
 4   motivo_paragem          496 non-null    str  
 5   data_inicio             8431 non-null   str  
 6   tipo_data_inicio        6369 non-null   str  
 7   data_fim_primaria       8428 non-null   str  
 8   tipo_data_fim_primaria  8427 non-null   str  
 9   data_fim                8377 non-null   str  
 10  tipo_data_fim           8374 non-null   str  
 11  participantes           8429 non-null   Int64
 12  fase                    8428 non-null   str  
 13  classe_patrocinador     8431 non-null   str  
dtypes: Int64(1), str(13)
memory usage: 930.5 KB


### Observações da inspeção

- **Dimensão:** a tabela tem 8 431 estudos e 14 colunas.
- **Validações:** o `nct_id` é único (não há estudos repetidos) e os 8 431 estudos são todos INTERVENTIONAL, o que confirma o filtro da extração.
- **Estudos parados sem motivo:** dos 536 estudos parados, 40 não indicam o motivo. Não há estudos não parados com motivo.
- **Tipo da data de início:** 24,5 % dos estudos não têm tipo (real ou estimada). Todos começaram entre 2010 e 2017; a partir de 2018 não há nenhum caso, o que sugere uma mudança na forma como os dados eram recolhidos.
- **Data de fim:** 54 estudos não têm data de fim. Outros 3 têm a data, mas não indicam se é real ou estimada.
- **Data de fim primária:** 3 estudos não têm esta data. Outro 1 tem a data, mas não indica se é real ou estimada.
- **Participantes e fase:** 2 estudos não indicam o número de participantes e 3 não indicam a fase.

## 3. Construir a tabela estudo_pais


In [21]:
linhas_pais = []
for estudo in estudos_brutos:                      
    protocolo = estudo.get("protocolSection", {})
    ident = protocolo.get("identificationModule", {})
    locais = protocolo.get("contactsLocationsModule", {})
    for centro in locais.get("locations", []):           
        linhas_pais.append({
            "nct_id": ident.get("nctId"),
            "pais": centro.get("country"),
        })

df_estudo_pais = pd.DataFrame(linhas_pais)
df_estudo_pais.shape

(29961, 2)

In [22]:
df_estudo_pais= df_estudo_pais.drop_duplicates().reset_index(drop=True)
df_estudo_pais.shape

(9270, 2)

In [23]:
df_estudo_pais["nct_id"].nunique()


7837

In [24]:
df_estudo_pais.isna().sum()

nct_id    0
pais      0
dtype: int64

In [25]:
df_estudo_pais["pais"].nunique()

107

In [26]:
print(df_estudo_pais["pais"].value_counts().sort_index().to_string())

pais
Albania                      1
Algeria                      1
Argentina                   55
Australia                  114
Austria                     46
Bangladesh                   4
Belgium                     84
Bolivia                      1
Bosnia and Herzegovina       1
Brazil                     211
Brunei                       1
Bulgaria                    23
Cameroon                     1
Canada                     407
Chile                       42
China                      419
Colombia                    20
Costa Rica                   1
Croatia                     14
Cuba                         2
Cyprus                       9
Czechia                     73
Denmark                    228
Ecuador                      2
Egypt                      161
Estonia                      3
Ethiopia                     2
Finland                     68
France                     341
Georgia                      6
Germany                    213
Greece                      72
Gre

In [27]:
total_estudos = df_estudos["nct_id"].nunique()
estudos_com_pais = df_estudo_pais["nct_id"].nunique()
estudos_sem_pais = total_estudos - estudos_com_pais
round(estudos_sem_pais / total_estudos * 100, 1)

7.0

In [28]:
sem_pais = ~df_estudos["nct_id"].isin(df_estudo_pais["nct_id"])
print(sem_pais.sum())
print(round(sem_pais.mean() * 100, 1))

594
7.0


In [29]:
df_estudo_pais["nct_id"].isin(df_estudos["nct_id"]).all()

np.True_

In [30]:
(df_estudo_pais["pais"] == "Portugal").sum()

np.int64(53)

### Observações da tabela estudo_pais

- **Dimensão:** 29 961 linhas (centros) antes de remover duplicados;9 270 pares estudo–país depois.
- **Estudos sem país:** 594 estudos (7 %) não indicam nenhum país.
- **Países:** 107 países diferentes. Não foram encontradas variantes de escrita.
- **Decisões:** 1 estudo indica "Serbia and Montenegro", nome anterior a 2006; mantido tal como registado. Os territórios (ex.: Puerto Rico, Hong Kong) foram mantidos separados, tal como no ClinicalTrials.gov.
- **Integridade:** todos os `nct_id` desta tabela existem na tabela `estudos`.
- **Portugal:** Participa em 53 estudos.

## 4. Construir a tabela estudo_intervencao

In [31]:
linhas_intervencao = []
for estudo in estudos_brutos:                      
    protocolo = estudo.get("protocolSection", {})
    ident = protocolo.get("identificationModule", {})
    modulo_interv = protocolo.get("armsInterventionsModule", {})
    for intervencao in modulo_interv.get("interventions", []):           
        linhas_intervencao.append({
            "nct_id": ident.get("nctId"),
            "tipo_intervencao": intervencao.get("type"),
        })

df_estudo_intervencao = pd.DataFrame(linhas_intervencao)
df_estudo_intervencao.shape

(16524, 2)

In [32]:
df_estudo_intervencao.isna().sum()

nct_id              0
tipo_intervencao    0
dtype: int64

In [33]:
df_estudo_intervencao["tipo_intervencao"].value_counts(dropna=False)

tipo_intervencao
BEHAVIORAL             5172
DRUG                   3607
OTHER                  3588
DIETARY_SUPPLEMENT     1974
DEVICE                  945
PROCEDURE               882
DIAGNOSTIC_TEST         150
BIOLOGICAL              110
COMBINATION_PRODUCT      66
GENETIC                  16
RADIATION                14
Name: count, dtype: int64

In [34]:
df_estudo_intervencao = df_estudo_intervencao.drop_duplicates().reset_index(drop=True)

In [35]:
type(df_estudo_intervencao)

pandas.DataFrame

In [36]:

df_estudo_intervencao["nct_id"].isin(df_estudos["nct_id"]).all()

np.True_

In [37]:
df_estudo_intervencao.shape

(9697, 2)

In [38]:
df_estudo_intervencao["tipo_intervencao"].value_counts()

tipo_intervencao
BEHAVIORAL             3107
OTHER                  2173
DRUG                   1790
DIETARY_SUPPLEMENT     1141
DEVICE                  683
PROCEDURE               590
DIAGNOSTIC_TEST          75
BIOLOGICAL               63
COMBINATION_PRODUCT      52
GENETIC                  13
RADIATION                10
Name: count, dtype: int64

In [39]:
sem_intervencao= ~df_estudos["nct_id"].isin(df_estudo_intervencao["nct_id"])


In [40]:
print(sem_intervencao.sum())
print(round(sem_intervencao.mean()*100,1))

0
0.0


In [41]:
df_estudo_intervencao["tipo_intervencao"].nunique() 

11

### Observações da tabela estudo_intervencao
- **Dimensão:** 16 524 linhas (intervenções) antes de remover duplicados; 9697 pares estudo–tipo depois.
- **Duplicados:** a maior descida foi no tipo DRUG (−50 %), o que é compatível com ensaios que registam o medicamento e o placebo, ambos como DRUG.
- **Estudos sem intervenção:** 0 (todos os estudos intervencionais indicam pelo menos uma intervenção).
- **Tipos:** 11 tipos, já padronizados pelo ClinicalTrials.gov; sem variantes de escrita.
- **Limitação:** o tipo OTHER (2 173 estudos) agrupa intervenções diversas e pouco específicas.
- **Nota para a análise:** um estudo pode ter vários tipos, por isso as contagens por tipo somam 9 697(mais do que os 8 431 estudos) e as percentagens por tipo não somam 100 %.
- **Integridade:** todos os `nct_id` desta tabela existem na tabela `estudos`.

## 5. Converter as datas

In [42]:
for coluna in ["data_inicio","data_fim_primaria", "data_fim"]:
    print(df_estudos[coluna].str.len().value_counts(dropna=False))
    print(df_estudos[coluna].str.contains("/", na=False).sum())
    print("----")

data_inicio
10    5765
7     2666
Name: count, dtype: int64
0
----
data_fim_primaria
10.0    5507
7.0     2921
NaN        3
Name: count, dtype: int64
0
----
data_fim
10.0    5484
7.0     2893
NaN       54
Name: count, dtype: int64
0
----


In [43]:
df_estudos.columns

Index(['nct_id', 'titulo', 'tipo_estudo', 'estado', 'motivo_paragem',
       'data_inicio', 'tipo_data_inicio', 'data_fim_primaria',
       'tipo_data_fim_primaria', 'data_fim', 'tipo_data_fim', 'participantes',
       'fase', 'classe_patrocinador'],
      dtype='str')

In [44]:
for coluna in ["data_inicio","data_fim_primaria", "data_fim"]:
    df_estudos[f"precisao_{coluna}"]=df_estudos[coluna].str.len().map({10:"dia",7:"mes"})
    so_mes=df_estudos[coluna].str.len()==7
    vazios_antes = df_estudos[coluna].isna().sum()  
    df_estudos.loc[so_mes,coluna]= df_estudos.loc[so_mes,coluna] + "-01"
    df_estudos[coluna]=pd.to_datetime(df_estudos[coluna],format="%Y-%m-%d")
    vazios_depois = df_estudos[coluna].isna().sum()
    print(coluna, vazios_antes, vazios_depois)


data_inicio 0 0
data_fim_primaria 3 3
data_fim 54 54


In [45]:
df_estudos.dtypes

nct_id                                   str
titulo                                   str
tipo_estudo                              str
estado                                   str
motivo_paragem                           str
data_inicio                   datetime64[us]
tipo_data_inicio                         str
data_fim_primaria             datetime64[us]
tipo_data_fim_primaria                   str
data_fim                      datetime64[us]
tipo_data_fim                            str
participantes                          Int64
fase                                     str
classe_patrocinador                      str
precisao_data_inicio                     str
precisao_data_fim_primaria               str
precisao_data_fim                        str
dtype: object

In [46]:
df_estudos["data_inicio"].min()

Timestamp('2010-01-01 00:00:00')

In [47]:
df_estudos["data_inicio"].max()

Timestamp('2025-12-31 00:00:00')

In [48]:
df_estudos["precisao_data_inicio"].value_counts(dropna=False)

precisao_data_inicio
dia    5765
mes    2666
Name: count, dtype: int64

In [49]:
df_estudos["ano_inicio"]= df_estudos["data_inicio"].dt.year

In [50]:
df_estudos["ano_inicio"]

0       2024
1       2012
2       2019
3       2010
4       2025
        ... 
8426    2024
8427    2011
8428    2024
8429    2014
8430    2021
Name: ano_inicio, Length: 8431, dtype: int32

In [51]:
df_estudos.dtypes

nct_id                                   str
titulo                                   str
tipo_estudo                              str
estado                                   str
motivo_paragem                           str
data_inicio                   datetime64[us]
tipo_data_inicio                         str
data_fim_primaria             datetime64[us]
tipo_data_fim_primaria                   str
data_fim                      datetime64[us]
tipo_data_fim                            str
participantes                          Int64
fase                                     str
classe_patrocinador                      str
precisao_data_inicio                     str
precisao_data_fim_primaria               str
precisao_data_fim                        str
ano_inicio                             int32
dtype: object

In [52]:
df_estudos["ano_inicio"].value_counts().sort_index()

ano_inicio
2010    359
2011    388
2012    441
2013    431
2014    437
2015    498
2016    519
2017    547
2018    553
2019    538
2020    436
2021    554
2022    580
2023    687
2024    702
2025    761
Name: count, dtype: int64

In [53]:
round((df_estudos["precisao_data_inicio"] == "mes").mean() * 100, 1)

np.float64(31.6)

In [54]:
round((df_estudos["precisao_data_fim_primaria"] == "mes").mean() * 100, 1)

np.float64(34.6)

In [55]:
round((df_estudos["precisao_data_fim"] == "mes").mean() * 100, 1)

np.float64(34.3)

### Observações da conversão das datas
- **Formatos:** só existem dois formatos (AAAA-MM-DD e AAAA-MM); não há datas com barras.
- **Datas só com mês:** completadas com o dia 1. A precisão original ficou guardada nas colunas `precisao_...`: início 31,6 %, fim primário 34,6 %, fim 34,3 %.
- **Conversão sem perdas:** os vazios antes e depois da conversão são iguais (0, 3 e 54).
- **Validação:** as datas de início estão entre 1 de janeiro de 2010 e 31 dezembro de 2025, o que confirma o filtro da extração.
- **Nova coluna:** `ano_inicio` (número inteiro).
- **Atenção na análise:** as datas com precisão "mes" não devem ser usadas em análises ao nível do dia (por exemplo, dia da semana).

## 6. Calcular a duração dos ensaios

In [56]:
df_estudos["tipo_data_inicio"].value_counts(dropna=False)

tipo_data_inicio
ACTUAL       5933
NaN          2062
ESTIMATED     436
Name: count, dtype: int64

In [57]:
df_estudos["tipo_data_fim_primaria"].value_counts(dropna=False)

tipo_data_fim_primaria
ACTUAL       6097
ESTIMATED    2330
NaN             4
Name: count, dtype: int64

In [58]:
DATA_EXTRACAO = pd.Timestamp("2026-10-12")

In [59]:
df_estudos[df_estudos["tipo_data_inicio"] == "ESTIMATED"]["estado"].value_counts()

estado
UNKNOWN                    224
NOT_YET_RECRUITING         121
WITHDRAWN                   68
RECRUITING                  19
SUSPENDED                    2
ENROLLING_BY_INVITATION      2
Name: count, dtype: int64

In [60]:
df_estudos[df_estudos["tipo_data_inicio"] == "ACTUAL"]["estado"].value_counts(dropna=False)


estado
COMPLETED                  3823
RECRUITING                  707
UNKNOWN                     599
ACTIVE_NOT_RECRUITING       426
TERMINATED                  242
WITHDRAWN                    73
ENROLLING_BY_INVITATION      49
SUSPENDED                    14
Name: count, dtype: int64

In [61]:
fim_real= df_estudos["tipo_data_fim_primaria"]=="ACTUAL"
fim_real.sum()

np.int64(6097)

In [62]:
inicio_real= df_estudos["tipo_data_inicio"]=="ACTUAL"
inicio_real.sum()

np.int64(5933)

In [63]:
fim_no_futuro= fim_real & (df_estudos["data_fim_primaria"] > DATA_EXTRACAO)
fim_no_futuro.sum()

np.int64(0)

In [64]:
fim_antes_inicio= df_estudos["data_fim_primaria"]< df_estudos["data_inicio"]
fim_antes_inicio.sum()

np.int64(0)

In [65]:
concluido_estimado= (df_estudos["estado"]=="COMPLETED") & (df_estudos["tipo_data_fim_primaria"]=="ESTIMATED")
concluido_estimado.sum()

np.int64(0)

In [66]:
#datas (sem olhar ao estado)
datas_ok = (fim_real & df_estudos["data_inicio"].notna()
            & ~fim_no_futuro & ~fim_antes_inicio)
datas_ok.sum()

np.int64(6097)

In [67]:
df_estudos[datas_ok]["estado"].value_counts()

estado
COMPLETED                  5430
TERMINATED                  317
ACTIVE_NOT_RECRUITING       150
UNKNOWN                      94
WITHDRAWN                    75
RECRUITING                   19
ENROLLING_BY_INVITATION       8
SUSPENDED                     4
Name: count, dtype: int64

In [68]:
chegou_resultado = df_estudos["estado"].isin(["COMPLETED", "ACTIVE_NOT_RECRUITING", "UNKNOWN"])
valida_pragmatica = datas_ok & chegou_resultado
valida_estrita = datas_ok & inicio_real & chegou_resultado

In [69]:
valida_pragmatica.sum()


np.int64(5674)

In [70]:
valida_estrita.sum()

np.int64(4033)

In [71]:
df_estudos["duracao_meses"] = ((df_estudos["data_fim_primaria"] - df_estudos["data_inicio"]).dt.days / 30.44).round(1)

In [72]:
df_estudos.loc[valida_pragmatica, "duracao_meses"].median()

np.float64(17.0)

In [73]:
df_estudos.loc[valida_estrita, "duracao_meses"].median()

np.float64(16.2)

In [74]:
df_estudos.loc[valida_pragmatica, "duracao_meses"].mean()

np.float64(22.743214663376808)

In [75]:
df_estudos.loc[~valida_pragmatica, "duracao_meses"] = None

In [76]:
df_estudos["duracao_meses"].describe()

count    5674.000000
mean       22.743215
std        19.593709
min         0.000000
25%         8.000000
50%        17.000000
75%        31.200000
max       188.200000
Name: duracao_meses, dtype: float64

In [77]:
(df_estudos["duracao_meses"] == 0).sum()

np.int64(20)

In [78]:
df_estudos.sort_values("duracao_meses", ascending=False)[["titulo", "duracao_meses"]].head()

,titulo,duracao_meses
2957,Role of Immune System in Obesity-related Infla...,188.2
5449,The Expecting Study,140.9
2773,An Exercise RCT Targeting African-American Wom...,140.0
3479,Starting Early Obesity Prevention Program,136.0
1946,An Endometrial Cancer Chemoprevention Study of...,132.0


In [79]:
zero = df_estudos["duracao_meses"] == 0
df_estudos[zero][["precisao_data_inicio", "precisao_data_fim_primaria"]].value_counts()

precisao_data_inicio  precisao_data_fim_primaria
mes                   mes                           11
dia                   dia                            9
Name: count, dtype: int64

In [80]:
df_estudos[zero]["titulo"]

1986            Inflammation Regulation in Obese Patients
2132    The Effect of Web-Based and Face-to-face Train...
2360    Mobile Health (M-health) Intervention to Reduc...
2997    Mindfulness Intervention for Overweight Primar...
3369      Swimming and Water Walking on Spirometry Values
3577    Effects of Sugary Drinks Counter-marketing Mes...
3760    Combined Training (Strength and Endurance) and...
3961    Saliva and Plasma Endocannabinoids Concentrati...
4159    Associations of Cognition, Fitness, and Body C...
4810    A Retrospective Analysis of the Effects of Red...
5667    Obesity in Schoolchildren of Basic Education -...
5693    Weight Stigma by Association in Parent-Child D...
6091    The Role of Empathy in the Links Between Binge...
6588       Effects of Radiofrequency on Located Adiposity
6951     Bioavailability of Blackberry Juice Anthocyanins
7583    Aquatic and Land Physical Training Effects on ...
7600    Breastfeeding in Infancy and Food Intake in Pr...
8131    Dairy 

### Observações da duração dos ensaios

- **Data de fim usada:** fim primário (tempo até ao resultado principal).
- **Tipos das datas:** início com 5 933 ACTUAL, 436 ESTIMATED e 2 062 sem tipo (todos de 2010 a 2017). Fim primário com 6 097 ACTUAL, 2 330 ESTIMATED e 4 sem tipo.
- **Validações:** 0 fins reais posteriores à extração (12/10/2026), 0 fins anteriores ao início e 0 estudos COMPLETED com o fim primário estimado.
- **Inconsistências:** 2 estudos com o início ESTIMATED mas o fim primário ACTUAL; 27 estudos ainda a recrutar (RECRUITING ou ENROLLING_BY_INVITATION) com o fim primário ACTUAL (excluídos).
- **Estudos incluídos:** fim primário ACTUAL, início com data e estado COMPLETED, ACTIVE_NOT_RECRUITING ou UNKNOWN (5 674 estudos). Os estudos interrompidos (TERMINATED, SUSPENDED, WITHDRAWN) foram excluídos, porque não chegaram ao resultado principal.
- **Resultado e sensibilidade:** mediana de 17,0 meses. Usando só inícios ACTUAL (4 033 estudos), a mediana é de 16,2 meses: o resultado é robusto à escolha do critério.
- **Distribuição:** metade dos ensaios demora entre 8,0 e 31,2 meses. A média (22,7 meses) é puxada por alguns ensaios muito longos (máximo de 188,2 meses).
- **Durações de 0 meses:** 20 estudos (11 com as duas datas só com mês e 9 com dia). Incluem estudos agudos (ex.: biodisponibilidade) e limitações de registo. Mantidos; impacto desprezável na mediana.